# Study 6 on Google Colab

Runs the locked `docs/PREREGISTRATION_STUDY6.md`. **No training:** five released detectors and Study 5's eight A0 checkpoints score frozen text sets (about 1–1.5 A100 hours).

**Before you start**
- *Runtime → Change runtime type → A100 GPU* (step 0 and step 4 need no GPU).
- Colab Secret (🔑) `HF_TOKEN` with notebook access on. **Never paste keys into cells.** Your Hugging Face account must have accepted Meta's licence for Llama Prompt Guard 2 (both sizes).
- The owner's Study 6 approval line must be in `AGENTS.md` (steps 1–4 only).

**Every time Colab (re)connects:** run the setup cell, then the step you are on. Finished work is skipped.

**Licence and blinding**
- Text stays in the private Drive folders. Never share it publicly.
- Do not open `study6/private_restricted_text_do_not_share/scores/` until step 4 finishes.
- Save labelled CSVs as **CSV UTF-8**, same file name, same folder.

In [ ]:
# One setup cell: secrets, Drive, repo, packages. Run this after every (re)connect.
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # Colab Secret; never printed
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
OUT = '/content/drive/MyDrive/study6'
S5 = '/content/drive/MyDrive/study5'
if not os.path.exists('/content/prompt_injection'):
    os.system('git clone -q -b claude/vigilant-fermi-r2ovyf https://github.com/dannyg26/prompt_injection /content/prompt_injection')
os.chdir('/content/prompt_injection')
!git pull -q origin claude/vigilant-fermi-r2ovyf
!git log --oneline -1
!pip install -q -r requirements-lock.txt
!pip install -q transformers==4.57.1 datasets==4.4.1 python-dotenv==1.0.1 pandas openpyxl sentencepiece protobuf tiktoken
!pip install -q --no-deps -e .
!nvidia-smi --query-gpu=name --format=csv,noheader

## 0. Second-annotator files (no scores; can run any time)

Writes `iaa_A2_blind.csv`, `iaa_B1_blind.csv`, `iaa_W_blind.csv` and `iaa_Opara_blind.csv` (50 rows each, **without your labels**) next to the original audits. Give them to the second annotator (a project member, privately) with `docs/AUDIT_RUBRIC.md`. They fill the 1/0 column without seeing your labels.

In [ ]:
!python scripts/export_iaa.py --drive /content/drive/MyDrive 2>&1 | grep -v -E 'oneDNN|cuda_|absl|cpu_feature|AVX|TF_ENABLE|All log messages|rebuild TensorFlow|Could not find cuda|cudart_stub|^I0000|^W0000|^E0000' ; echo "EXIT DONE"

## 1. Prepare: attack rows, substitution, audit export; no scores (a few minutes)

In [ ]:
!PYTHONPATH=src python scripts/run_study6.py --out "$OUT" --study5 "$S5" --stage prepare 2>&1 | grep -v -E 'oneDNN|cuda_|absl|cpu_feature|AVX|TF_ENABLE|All log messages|rebuild TensorFlow|Could not find cuda|cudart_stub|^I0000|^W0000|^E0000' ; echo "EXIT DONE"

## 2. Score (about 1–1.5 hours; resumable). No results yet.

In [ ]:
!PYTHONPATH=src python scripts/run_study6.py --out "$OUT" --study5 "$S5" --stage score 2>&1 | grep -v -E 'oneDNN|cuda_|absl|cpu_feature|AVX|TF_ENABLE|All log messages|rebuild TensorFlow|Could not find cuda|cudart_stub|^I0000|^W0000|^E0000' ; echo "EXIT DONE"

## 3. Label the substitution audit BEFORE analysis

Open `study6/private_restricted_text_do_not_share/audit_SUB_blind.csv` (50 rows). For each row, put **1** in `same_attack_intent (1/0)` if the substituted text still asks for the same attack as the original (same goal, e.g. still tries to override instructions or extract the hidden prompt), else **0** (also 0 if unsure). Save as CSV UTF-8, same name. Count the 1s.

## 4. Analysis (refuses to run without the audit count)

In [ ]:
SUB = '__'   # replace __ with your count of 1s (out of 50)
!PYTHONPATH=src python scripts/run_study6.py --out "$OUT" --study5 "$S5" --stage analyze --audit "SUB=$SUB" 2>&1 | grep -v -E 'oneDNN|cuda_|absl|cpu_feature|AVX|TF_ENABLE|All log messages|rebuild TensorFlow|Could not find cuda|cudart_stub|^I0000|^W0000|^E0000' ; echo "EXIT DONE"

## 5. Agreement (after the second annotator has filled all four iaa_ files)

In [ ]:
!PYTHONPATH=src python scripts/run_study6.py --out "$OUT" --study5 "$S5" --stage iaa 2>&1 | grep -v -E 'oneDNN|cuda_|absl|cpu_feature|AVX|TF_ENABLE|All log messages|rebuild TensorFlow|Could not find cuda|cudart_stub|^I0000|^W0000|^E0000' ; echo "EXIT DONE"

## 6. Download the result files and send them back

In [ ]:
from google.colab import files
for name in ('study6_results.json', 'prep_manifest.json', 'detector_revisions.json', 'iaa_results.json'):
    path = f'{OUT}/public/{name}'
    if os.path.exists(path):
        files.download(path)
    else:
        print('not yet:', name)